# 4. User-Query Inference Flow

This notebook loads the saved retrieval artifacts and runs true **query-to-document inference** for a small set of user-like queries. It is intentionally separate from evaluation: the goal here is to inspect the inference path, compare top-3 outputs across methods, and keep the code shape reusable for future retrieval-run generation.

Unlike the older recipe-to-recipe evaluation style, this notebook does not use `source_doc_id` or a row from a precomputed similarity matrix as the query. Every method receives a raw query string, applies the same normalization used in training where applicable, computes query-to-document scores, and returns the top results.

## 1. Setup

The path resolver below works whether the notebook is launched from the repository root or from the notebook directory. All model artifacts are expected under `Finalproject/notebooks/Saved_models` after rerunning `2_Content_based_methods.ipynb` and `3_Model_based_methods.ipynb`.

In [19]:
from pathlib import Path
import ast
import json
import math
import os
import pickle
import re
import warnings

import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel

warnings.filterwarnings("ignore")

try:
    import faiss
except ImportError:
    faiss = None

try:
    from sentence_transformers import SentenceTransformer
except ImportError:
    SentenceTransformer = None

try:
    from rank_bm25 import BM25Okapi
except ImportError:
    BM25Okapi = None

In [20]:
def find_finalproject_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "data" / "all_recipes_final.csv").exists():
            return candidate
        if (candidate / "Finalproject" / "data" / "all_recipes_final.csv").exists():
            return candidate / "Finalproject"
    raise FileNotFoundError("Could not locate Finalproject/data/all_recipes_final.csv")


FINALPROJECT_ROOT = find_finalproject_root()
DATA_PATH = FINALPROJECT_ROOT / "data" / "all_recipes_final.csv"
NOTEBOOKS_ROOT = FINALPROJECT_ROOT / "notebooks"
SAVED_MODELS_DIR = NOTEBOOKS_ROOT / "Saved_models"
RAREC_DIR = NOTEBOOKS_ROOT / "RA_Rec"
DEFAULT_QUERY_FILE = NOTEBOOKS_ROOT / "rec_and_eval" / "groundtruth_outputs" / "queries_500_title_as_query.csv"

print("Finalproject root:", FINALPROJECT_ROOT)
print("Data path:", DATA_PATH)
print("Saved models:", SAVED_MODELS_DIR)
print("RARec directory:", RAREC_DIR)

Finalproject root: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject
Data path: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\data\all_recipes_final.csv
Saved models: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\Saved_models
RARec directory: C:\Users\tientm1\DS300-UIT-RecommenderSystem\Finalproject\notebooks\RA_Rec


## 2. Load Recipes and Query Samples

The query file is used only to pick a few sample queries for inspection. Because the future benchmark is intended to represent user queries, the notebook also provides `USER_QUERY`; edit that value and rerun the inference cells to inspect your own query.

In [21]:
recipes = pd.read_csv(DATA_PATH)
recipes["doc_id"] = recipes.index.astype(int)

for column_name in [
    "title",
    "description",
    "ingredients",
    "ingredients_normalized",
    "step",
    "type_of_food",
    "cook_time",
    "num_of_people",
    "calories",
    "link",
    "note",
]:
    if column_name in recipes.columns:
        recipes[column_name] = recipes[column_name].fillna("")

print("Number of recipes:", len(recipes))
recipes[["doc_id", "title", "type_of_food", "cook_time"]].head()

Number of recipes: 10263


,doc_id,title,type_of_food,cook_time
0,0,Cách muối dưa hành truyền thống,Món Tết,45 phút
1,1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,50 phút
2,2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,100 phút
3,3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,60 phút
4,4,Chả bì ớt xiêm xanh,Món Tết,60 phút


In [22]:
QUERY_FILE_PATH = DEFAULT_QUERY_FILE
NUM_SAMPLE_QUERIES = 10
QUERY_SAMPLE_RANDOM_SEED = 42

if QUERY_FILE_PATH.exists():
    query_samples = pd.read_csv(QUERY_FILE_PATH)
    if "query_text" not in query_samples.columns:
        raise ValueError(f"Query file must contain a query_text column: {QUERY_FILE_PATH}")
    query_samples = (
        query_samples[["query_text"]]
        .dropna()
        .drop_duplicates()
        .sample(n=min(NUM_SAMPLE_QUERIES, len(query_samples)), random_state=QUERY_SAMPLE_RANDOM_SEED)
        .reset_index(drop=True)
    )
else:
    query_samples = pd.DataFrame({
        "query_text": [
            "món kho có thịt heo cho bữa cơm gia đình",
            "món chay ít dầu dễ nấu",
            "canh thanh mát cho ngày nóng",
            "món gà nướng thơm cay",
            "bữa sáng nhanh với trứng",
            "món ăn Tết truyền thống",
            "món hải sản hấp đơn giản",
            "salad rau củ nhẹ bụng",
            "món bò xào đậm vị",
            "chè ngọt ăn tráng miệng",
        ]
    })

print("Sample queries:")
query_samples

Sample queries:


,query_text
0,Canh bí ngòi nấu tôm theo công thức của Chef T...
1,Bánh bông lan flan bằng nồi chiên không dầu th...
2,Cháo rau muống cá thu cho bé thơm ngon hấp dẫn...
3,Bánh quy nhân kem trứng thơm béo lạ mắt đơn giản
4,"Canh cua nấu mướp siêu hấp dẫn, đậm đà cho bữa..."
5,Phi lê cá rô phi chiên giòn siêu dễ bằng chảo ...
6,Cách làm pudding xoài cho bé ăn dặm thơm ...
7,Cháo khoai gạo lứt dành cho người đau dạ dày
8,Dưa gang muối chua vàng giòn thơm ngon không b...
9,Bò bía mặn chấm tương đen đậu phộng béo ngậy n...


In [23]:
# Edit this value when you want to inspect a custom user query.
USER_QUERY = "món kho có thịt heo ăn với cơm"

queries_to_inspect = [USER_QUERY] + query_samples["query_text"].tolist()
queries_to_inspect = list(dict.fromkeys([query.strip() for query in queries_to_inspect if str(query).strip()]))

print("Number of queries to inspect:", len(queries_to_inspect))
queries_to_inspect[:5]

Number of queries to inspect: 11


['món kho có thịt heo ăn với cơm',
 'Canh bí ngòi nấu tôm theo công thức của Chef Tuyết Phạm',
 'Bánh bông lan flan bằng nồi chiên không dầu thơm ngon mềm mịn',
 'Cháo rau muống cá thu cho bé thơm ngon hấp dẫn đơn giản',
 'Bánh quy nhân kem trứng thơm béo lạ mắt đơn giản']

## 3. Training-Compatible Normalization

These helpers mirror the preprocessing choices used by the training notebooks. TF-IDF and Ingredient TF-IDF receive cleaned query text. Keyword retrieval uses the same title-oriented keyword extraction logic as the content-based notebook. SBERT and RARec receive the raw user query, matching their original query-time behavior.

In [24]:
def parse_list_string(value):
    if pd.isna(value) or value == "[]":
        return []
    try:
        parsed_value = ast.literal_eval(str(value))
        return parsed_value if isinstance(parsed_value, list) else []
    except (ValueError, SyntaxError):
        return []


def parse_set_string(value):
    if pd.isna(value) or value == "set()":
        return []
    try:
        parsed_value = ast.literal_eval(str(value))
        if isinstance(parsed_value, (set, list, tuple)):
            return list(parsed_value)
        return []
    except (ValueError, SyntaxError):
        return []


def clean_text(text):
    if pd.isna(text):
        return ""
    normalized_text = str(text).lower()
    normalized_text = re.sub(r"[^\w\s\u00C0-\u1EF9]", " ", normalized_text)
    normalized_text = re.sub(r"\s+", " ", normalized_text).strip()
    return normalized_text


def normalize_query_for_tfidf(query: str) -> str:
    return clean_text(query)


def normalize_query_for_ingredient_tfidf(query: str) -> str:
    return clean_text(query)

In [25]:
COOKING_METHODS = [
    "xào", "nướng", "luộc", "chiên", "hấp", "kho", "rim", "rang",
    "canh", "súp", "cháo", "gỏi", "nộm", "salad", "bún", "phở",
    "mì", "cơm", "bánh", "chè", "sinh tố",
]


def extract_keywords(title, ingredients_normalized_list=None):
    keywords = set()
    title_text = clean_text(title)

    for method in COOKING_METHODS:
        if method in title_text:
            keywords.add(method)

    for word in title_text.split():
        if len(word) > 2:
            keywords.add(word)

    if ingredients_normalized_list:
        for ingredient in ingredients_normalized_list:
            cleaned_ingredient = clean_text(str(ingredient))
            if len(cleaned_ingredient) > 2:
                keywords.add(cleaned_ingredient)

    return keywords


def jaccard_similarity(left_set, right_set):
    if not left_set and not right_set:
        return 0.0
    union_size = len(left_set.union(right_set))
    if union_size == 0:
        return 0.0
    return len(left_set.intersection(right_set)) / union_size


recipes["ingredients_normalized_list"] = recipes["ingredients_normalized"].apply(parse_set_string)
recipes["keyword_set"] = recipes.apply(
    lambda row: extract_keywords(row["title"], row["ingredients_normalized_list"]),
    axis=1,
)

## 4. Load Retrieval Artifacts

Each loader returns `None` if its artifacts are missing. That makes the notebook usable while models are still being regenerated: available methods can run, and missing methods are listed clearly.

In [26]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def load_pickle(path: Path):
    with open(path, "rb") as file:
        return pickle.load(file)


def missing_files(paths):
    return [str(path) for path in paths if not Path(path).exists()]


loaded_artifacts = {}
missing_artifacts = {}

In [27]:
# TF-IDF artifacts
tfidf_dir = SAVED_MODELS_DIR / "TFIDF"
tfidf_required_files = [
    tfidf_dir / "tfidf_vectorizer.pkl",
    tfidf_dir / "tfidf_matrix.pkl",
    tfidf_dir / "tfidf_similarity.npy",
    tfidf_dir / "metadata.json",
]
missing = missing_files(tfidf_required_files)
if missing:
    missing_artifacts["TFIDF"] = missing
else:
    loaded_artifacts["TFIDF"] = {
        "vectorizer": load_pickle(tfidf_dir / "tfidf_vectorizer.pkl"),
        "matrix": load_pickle(tfidf_dir / "tfidf_matrix.pkl"),
        "similarity": np.load(tfidf_dir / "tfidf_similarity.npy"),
        "metadata": load_json(tfidf_dir / "metadata.json"),
    }

# Ingredient TF-IDF artifacts
ingredient_dir = SAVED_MODELS_DIR / "Ingredient_TFIDF"
ingredient_required_files = [
    ingredient_dir / "ingredient_tfidf_vectorizer.pkl",
    ingredient_dir / "ingredient_tfidf_matrix.pkl",
    ingredient_dir / "ingredient_tfidf_similarity.npy",
    ingredient_dir / "metadata.json",
]
missing = missing_files(ingredient_required_files)
if missing:
    missing_artifacts["Ingredient_TFIDF"] = missing
else:
    loaded_artifacts["Ingredient_TFIDF"] = {
        "vectorizer": load_pickle(ingredient_dir / "ingredient_tfidf_vectorizer.pkl"),
        "matrix": load_pickle(ingredient_dir / "ingredient_tfidf_matrix.pkl"),
        "similarity": np.load(ingredient_dir / "ingredient_tfidf_similarity.npy"),
        "metadata": load_json(ingredient_dir / "metadata.json"),
    }

# Hybrid content artifacts
hybrid_content_dir = SAVED_MODELS_DIR / "Hybrid"
hybrid_content_required_files = [
    hybrid_content_dir / "hybrid_similarity.npy",
    hybrid_content_dir / "metadata.json",
]
missing = missing_files(hybrid_content_required_files)
if missing:
    missing_artifacts["Hybrid"] = missing
else:
    loaded_artifacts["Hybrid"] = {
        "similarity": np.load(hybrid_content_dir / "hybrid_similarity.npy"),
        "metadata": load_json(hybrid_content_dir / "metadata.json"),
    }

# Keyword artifacts
keyword_dir = SAVED_MODELS_DIR / "Keyword"
keyword_required_files = [
    keyword_dir / "keyword_similarity.npy",
    keyword_dir / "metadata.json",
]
missing = missing_files(keyword_required_files)
if missing:
    missing_artifacts["Keyword"] = missing
else:
    loaded_artifacts["Keyword"] = {
        "similarity": np.load(keyword_dir / "keyword_similarity.npy"),
        "metadata": load_json(keyword_dir / "metadata.json"),
    }

print("Loaded artifact groups:", sorted(loaded_artifacts.keys()))
print("Missing artifact groups:", sorted(missing_artifacts.keys()))
missing_artifacts

Loaded artifact groups: ['Hybrid', 'Ingredient_TFIDF', 'Keyword', 'TFIDF']
Missing artifact groups: []


{}

In [28]:
# SBERT / FAISS artifacts
sbert_dir = SAVED_MODELS_DIR / "SBERT_FAISS"
sbert_required_files = [
    sbert_dir / "recipe_embeddings.npy",
    sbert_dir / "model_info.json",
]
if faiss is not None:
    sbert_required_files.append(sbert_dir / "faiss_index.bin")

missing = missing_files(sbert_required_files)
if missing or SentenceTransformer is None:
    missing_artifacts["SBERT_FAISS"] = missing
    if SentenceTransformer is None:
        missing_artifacts["SBERT_FAISS"].append("sentence_transformers is not installed")
else:
    sbert_model_info = load_json(sbert_dir / "model_info.json")
    sbert_model = SentenceTransformer(sbert_model_info["model_name"])
    loaded_artifacts["SBERT_FAISS"] = {
        "model": sbert_model,
        "embeddings": np.load(sbert_dir / "recipe_embeddings.npy"),
        "model_info": sbert_model_info,
        "faiss_index": faiss.read_index(str(sbert_dir / "faiss_index.bin")) if faiss is not None else None,
    }

# Hybrid TF-IDF + SBERT artifacts
hybrid_tfidf_sbert_dir = SAVED_MODELS_DIR / "Hybrid_TFIDF_SBERT"
hybrid_tfidf_sbert_required_files = [
    hybrid_tfidf_sbert_dir / "sbert_embeddings.npy",
    hybrid_tfidf_sbert_dir / "config.json",
]
missing = missing_files(hybrid_tfidf_sbert_required_files)
if missing:
    missing_artifacts["Hybrid_TFIDF_SBERT"] = missing
else:
    loaded_artifacts["Hybrid_TFIDF_SBERT"] = {
        "sbert_embeddings": np.load(hybrid_tfidf_sbert_dir / "sbert_embeddings.npy"),
        "config": load_json(hybrid_tfidf_sbert_dir / "config.json"),
    }

print("Loaded artifact groups:", sorted(loaded_artifacts.keys()))
print("Missing artifact groups:", sorted(missing_artifacts.keys()))

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13311.64it/s]


Loaded artifact groups: ['Hybrid', 'Hybrid_TFIDF_SBERT', 'Ingredient_TFIDF', 'Keyword', 'SBERT_FAISS', 'TFIDF']
Missing artifact groups: []


In [29]:
# RARec Late Fusion artifacts
rarec_embeddings_path = RAREC_DIR / "recipes_embeddings_list.pkl"
if not rarec_embeddings_path.exists() or SentenceTransformer is None:
    missing_artifacts["RARec_Late_Fusion"] = []
    if not rarec_embeddings_path.exists():
        missing_artifacts["RARec_Late_Fusion"].append(str(rarec_embeddings_path))
    if SentenceTransformer is None:
        missing_artifacts["RARec_Late_Fusion"].append("sentence_transformers is not installed")
else:
    rarec_model_name = (
        loaded_artifacts.get("SBERT_FAISS", {})
        .get("model_info", {})
        .get("model_name", "keepitreal/vietnamese-sbert")
    )
    rarec_model = loaded_artifacts.get("SBERT_FAISS", {}).get("model")
    if rarec_model is None:
        rarec_model = SentenceTransformer(rarec_model_name)

    loaded_artifacts["RARec_Late_Fusion"] = {
        "model": rarec_model,
        "recipes_embeddings_list": load_pickle(rarec_embeddings_path),
        "model_name": rarec_model_name,
    }

print("Loaded artifact groups:", sorted(loaded_artifacts.keys()))
print("Missing artifact groups:", sorted(missing_artifacts.keys()))

Loaded artifact groups: ['Hybrid', 'Hybrid_TFIDF_SBERT', 'Ingredient_TFIDF', 'Keyword', 'RARec_Late_Fusion', 'SBERT_FAISS', 'TFIDF']
Missing artifact groups: []


## 5. Query-to-Document Scoring Functions

All functions return the same compact schema: `method`, `rank`, `doc_id`, `score`, `title`, and a few recipe metadata fields. This keeps the notebook inspectable now and makes the code easier to migrate into an evaluation runner later.

In [30]:
RESULT_COLUMNS = [
    "method",
    "rank",
    "doc_id",
    "score",
    "title",
    "type_of_food",
    "cook_time",
    "calories",
    "link",
]


def top_indices_from_scores(scores, top_k=3):
    scores = np.asarray(scores, dtype=float)
    if len(scores) == 0:
        return np.array([], dtype=int)
    candidate_count = min(top_k, len(scores))
    top_indices = np.argpartition(scores, -candidate_count)[-candidate_count:]
    top_indices = top_indices[np.argsort(scores[top_indices])[::-1]]
    return top_indices


def format_results(method_name, scores, top_k=3):
    top_indices = top_indices_from_scores(scores, top_k=top_k)
    rows = []
    for rank, doc_id in enumerate(top_indices, start=1):
        recipe = recipes.iloc[int(doc_id)]
        rows.append({
            "method": method_name,
            "rank": rank,
            "doc_id": int(doc_id),
            "score": float(scores[int(doc_id)]),
            "title": recipe.get("title", ""),
            "type_of_food": recipe.get("type_of_food", ""),
            "cook_time": recipe.get("cook_time", ""),
            "calories": recipe.get("calories", ""),
            "link": recipe.get("link", ""),
        })
    return pd.DataFrame(rows, columns=RESULT_COLUMNS)

In [31]:
def infer_tfidf(query, top_k=3):
    artifacts = loaded_artifacts.get("TFIDF")
    if artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    normalized_query = normalize_query_for_tfidf(query)
    query_vector = artifacts["vectorizer"].transform([normalized_query])
    scores = linear_kernel(query_vector, artifacts["matrix"]).flatten()
    return format_results("TFIDF", scores, top_k=top_k)


def infer_ingredient_tfidf(query, top_k=3):
    artifacts = loaded_artifacts.get("Ingredient_TFIDF")
    if artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    normalized_query = normalize_query_for_ingredient_tfidf(query)
    query_vector = artifacts["vectorizer"].transform([normalized_query])
    scores = linear_kernel(query_vector, artifacts["matrix"]).flatten()
    return format_results("Ingredient_TFIDF", scores, top_k=top_k)


def infer_keyword(query, top_k=3):
    query_keywords = extract_keywords(query, [])
    scores = np.array([
        jaccard_similarity(query_keywords, recipe_keywords)
        for recipe_keywords in recipes["keyword_set"]
    ])
    return format_results("Keyword", scores, top_k=top_k)


def infer_hybrid_content(query, top_k=3, tfidf_weight=0.4, ingredient_tfidf_weight=0.6):
    tfidf_artifacts = loaded_artifacts.get("TFIDF")
    ingredient_artifacts = loaded_artifacts.get("Ingredient_TFIDF")
    if tfidf_artifacts is None or ingredient_artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    tfidf_query = tfidf_artifacts["vectorizer"].transform([normalize_query_for_tfidf(query)])
    ingredient_query = ingredient_artifacts["vectorizer"].transform([normalize_query_for_ingredient_tfidf(query)])

    tfidf_scores = linear_kernel(tfidf_query, tfidf_artifacts["matrix"]).flatten()
    ingredient_scores = linear_kernel(ingredient_query, ingredient_artifacts["matrix"]).flatten()

    total_weight = tfidf_weight + ingredient_tfidf_weight
    scores = (tfidf_weight / total_weight) * tfidf_scores + (ingredient_tfidf_weight / total_weight) * ingredient_scores
    return format_results("Hybrid_Content", scores, top_k=top_k)

In [32]:
def infer_sbert_faiss(query, top_k=3):
    artifacts = loaded_artifacts.get("SBERT_FAISS")
    if artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    query_embedding = artifacts["model"].encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )

    if artifacts.get("faiss_index") is not None:
        scores, indices = artifacts["faiss_index"].search(query_embedding, top_k)
        rows = []
        for rank, (doc_id, score) in enumerate(zip(indices[0], scores[0]), start=1):
            recipe = recipes.iloc[int(doc_id)]
            rows.append({
                "method": "SBERT_FAISS",
                "rank": rank,
                "doc_id": int(doc_id),
                "score": float(score),
                "title": recipe.get("title", ""),
                "type_of_food": recipe.get("type_of_food", ""),
                "cook_time": recipe.get("cook_time", ""),
                "calories": recipe.get("calories", ""),
                "link": recipe.get("link", ""),
            })
        return pd.DataFrame(rows, columns=RESULT_COLUMNS)

    scores = cosine_similarity(query_embedding, artifacts["embeddings"]).flatten()
    return format_results("SBERT_FAISS", scores, top_k=top_k)


def infer_hybrid_tfidf_sbert(query, top_k=3):
    tfidf_artifacts = loaded_artifacts.get("TFIDF")
    hybrid_artifacts = loaded_artifacts.get("Hybrid_TFIDF_SBERT")
    sbert_artifacts = loaded_artifacts.get("SBERT_FAISS")
    if tfidf_artifacts is None or hybrid_artifacts is None or sbert_artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    alpha = float(hybrid_artifacts["config"].get("alpha", 0.5))
    tfidf_query = tfidf_artifacts["vectorizer"].transform([normalize_query_for_tfidf(query)])
    tfidf_scores = linear_kernel(tfidf_query, tfidf_artifacts["matrix"]).flatten()

    query_embedding = sbert_artifacts["model"].encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    sbert_scores = cosine_similarity(query_embedding, hybrid_artifacts["sbert_embeddings"]).flatten()

    scores = alpha * tfidf_scores + (1.0 - alpha) * sbert_scores
    return format_results("Hybrid_TFIDF_SBERT", scores, top_k=top_k)

In [33]:
def infer_rarec_late_fusion(query, top_k=3):
    artifacts = loaded_artifacts.get("RARec_Late_Fusion")
    if artifacts is None:
        return pd.DataFrame(columns=RESULT_COLUMNS + ["max_similarity", "min_similarity", "num_sentences"])

    model = artifacts["model"]
    recipes_embeddings_list = artifacts["recipes_embeddings_list"]

    query_embedding = model.encode([query], convert_to_numpy=True)
    query_norm = np.linalg.norm(query_embedding)
    if query_norm == 0:
        return pd.DataFrame(columns=RESULT_COLUMNS + ["max_similarity", "min_similarity", "num_sentences"])
    query_embedding = query_embedding / query_norm

    recipe_scores = []
    for recipe_idx, dish_embeddings in enumerate(recipes_embeddings_list):
        if dish_embeddings is None or len(dish_embeddings) == 0:
            continue
        dish_embeddings = np.asarray(dish_embeddings)
        norms = np.linalg.norm(dish_embeddings, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        normalized_dish_embeddings = dish_embeddings / norms
        similarities = np.dot(normalized_dish_embeddings, query_embedding.T).flatten()
        recipe_scores.append({
            "doc_id": int(recipe_idx),
            "score": float(np.mean(similarities)),
            "max_similarity": float(np.max(similarities)),
            "min_similarity": float(np.min(similarities)),
            "num_sentences": int(len(similarities)),
        })

    recipe_scores = sorted(recipe_scores, key=lambda item: item["score"], reverse=True)[:top_k]
    rows = []
    for rank, item in enumerate(recipe_scores, start=1):
        recipe = recipes.iloc[item["doc_id"]]
        rows.append({
            "method": "RARec_Late_Fusion",
            "rank": rank,
            "doc_id": item["doc_id"],
            "score": item["score"],
            "title": recipe.get("title", ""),
            "type_of_food": recipe.get("type_of_food", ""),
            "cook_time": recipe.get("cook_time", ""),
            "calories": recipe.get("calories", ""),
            "link": recipe.get("link", ""),
            "max_similarity": item["max_similarity"],
            "min_similarity": item["min_similarity"],
            "num_sentences": item["num_sentences"],
        })
    return pd.DataFrame(rows)

In [34]:
def tokenize_for_bm25(text):
    normalized_text = clean_text(text)
    return normalized_text.split()


def build_bm25_document_text(row):
    text_parts = [
        row.get("title", ""),
        row.get("type_of_food", ""),
        row.get("description", ""),
        " ".join(parse_set_string(row.get("ingredients_normalized", ""))),
        " ".join(parse_list_string(row.get("ingredients", ""))),
        " ".join(parse_list_string(row.get("step", ""))),
    ]
    return " ".join(part for part in text_parts if str(part).strip())


if BM25Okapi is not None:
    bm25_corpus_tokens = recipes.apply(build_bm25_document_text, axis=1).apply(tokenize_for_bm25).tolist()
    bm25_model = BM25Okapi(bm25_corpus_tokens)
else:
    bm25_model = None
    missing_artifacts["BM25"] = ["rank_bm25 is not installed"]


def infer_bm25(query, top_k=3):
    if bm25_model is None:
        return pd.DataFrame(columns=RESULT_COLUMNS)
    query_tokens = tokenize_for_bm25(query)
    scores = bm25_model.get_scores(query_tokens)
    return format_results("BM25", scores, top_k=top_k)

## 6. Compare Top-3 Results

Run `compare_methods_for_query(USER_QUERY)` for a custom query, or use the loop below to inspect about ten sampled queries. The result table is intentionally compact; use the links and `doc_id` values to inspect recipes in more detail.

In [35]:
INFERENCE_METHODS = [
    infer_bm25,
    infer_tfidf,
    infer_ingredient_tfidf,
    infer_keyword,
    infer_hybrid_content,
    infer_sbert_faiss,
    infer_hybrid_tfidf_sbert,
    infer_rarec_late_fusion,
]


def compare_methods_for_query(query, top_k=3):
    frames = []
    for inference_method in INFERENCE_METHODS:
        try:
            method_results = inference_method(query, top_k=top_k)
            if len(method_results) > 0:
                frames.append(method_results)
        except Exception as error:
            print(f"{inference_method.__name__} failed: {error}")

    if not frames:
        return pd.DataFrame(columns=RESULT_COLUMNS)

    combined_results = pd.concat(frames, ignore_index=True)
    return combined_results


custom_query_results = compare_methods_for_query(USER_QUERY, top_k=3)
print("USER_QUERY:", USER_QUERY)
custom_query_results

USER_QUERY: món kho có thịt heo ăn với cơm


,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,6044,0.429079,Thịt heo quay kho tiêu đậm đà đưa cơm chuẩn vị...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
1,TFIDF,2,6449,0.388355,Thịt kho chua ngọt thơm ngon đậm đà cực đưa cơm,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
2,TFIDF,3,6445,0.377631,Thịt nạc kho tiêu nước sệt ngon miệng đậm đà d...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,5207,0.215885,Bầu xào thịt heo thanh vị bắt cơm ngay tại nhà,Món xào,10 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,4979,0.215758,Cháo từ cơm nguội nhanh chóng tiện lợi cực đơn...,Món cháo,15 phút,,https://www.dienmayxanh.com/vao-bep/cach-nau-c...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,7428,0.209397,Thịt heo chiên giòn không cần bột siêu hấp dẫn...,Món chiên,7 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
6,Keyword,1,1040,0.272727,Thịt heo kho cùi dừa,Món chính,45phút,,https://vncooking.com/cong-thuc/thit-heo-kho-c...,NaN,NaN,NaN
7,Keyword,2,6044,0.250000,Thịt heo quay kho tiêu đậm đà đưa cơm chuẩn vị...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
8,Keyword,3,955,0.250000,Thịt gà kho gừng,Món chính,30phút,,https://vncooking.com/cong-thuc/thit-ga-kho-gu...,NaN,NaN,NaN
9,Hybrid_Content,1,7428,0.261640,Thịt heo chiên giòn không cần bột siêu hấp dẫn...,Món chiên,7 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN


In [36]:
for query in queries_to_inspect[:NUM_SAMPLE_QUERIES]:
    print("=" * 120)
    print("Query:", query)
    display(compare_methods_for_query(query, top_k=3))

Query: món kho có thịt heo ăn với cơm


,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,6044,0.429079,Thịt heo quay kho tiêu đậm đà đưa cơm chuẩn vị...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
1,TFIDF,2,6449,0.388355,Thịt kho chua ngọt thơm ngon đậm đà cực đưa cơm,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
2,TFIDF,3,6445,0.377631,Thịt nạc kho tiêu nước sệt ngon miệng đậm đà d...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,5207,0.215885,Bầu xào thịt heo thanh vị bắt cơm ngay tại nhà,Món xào,10 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,4979,0.215758,Cháo từ cơm nguội nhanh chóng tiện lợi cực đơn...,Món cháo,15 phút,,https://www.dienmayxanh.com/vao-bep/cach-nau-c...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,7428,0.209397,Thịt heo chiên giòn không cần bột siêu hấp dẫn...,Món chiên,7 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
6,Keyword,1,1040,0.272727,Thịt heo kho cùi dừa,Món chính,45phút,,https://vncooking.com/cong-thuc/thit-heo-kho-c...,NaN,NaN,NaN
7,Keyword,2,6044,0.250000,Thịt heo quay kho tiêu đậm đà đưa cơm chuẩn vị...,Món kho,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
8,Keyword,3,955,0.250000,Thịt gà kho gừng,Món chính,30phút,,https://vncooking.com/cong-thuc/thit-ga-kho-gu...,NaN,NaN,NaN
9,Hybrid_Content,1,7428,0.261640,Thịt heo chiên giòn không cần bột siêu hấp dẫn...,Món chiên,7 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN


Query: Canh bí ngòi nấu tôm theo công thức của Chef Tuyết Phạm


,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,394,0.444971,Canh bí ngòi nấu tôm theo công thức của Chef T...,Món ngon hàng ngày,,,https://vnexpress.net/canh-bi-ngoi-nau-tom-the...,NaN,NaN,NaN
1,TFIDF,2,283,0.373388,Canh tôm nấu bí,Món ngon hàng ngày,,,https://vnexpress.net/canh-tom-nau-bi-4311354....,NaN,NaN,NaN
2,TFIDF,3,6026,0.370151,Cách sơ chế bông bí nụ giòn ngọt và xanh mướt ...,Món xào,2 phút,,https://www.dienmayxanh.com/vao-bep/cach-so-ch...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,4593,0.399539,"Món rau củ hầm Ratatouille đơn giản, thơm ngon...",Món nướng,90 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-m...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,2917,0.346352,Bánh bí ngòi Hàn Quốc đơn giản lạ miệng ăn hoà...,Món bánh,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,5610,0.345548,"Bí ngòi xào trứng đơn giản, ngon miệng bằng ch...",Món xào,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
6,Keyword,1,394,0.611111,Canh bí ngòi nấu tôm theo công thức của Chef T...,Món ngon hàng ngày,,,https://vnexpress.net/canh-bi-ngoi-nau-tom-the...,NaN,NaN,NaN
7,Keyword,2,924,0.187500,Canh bí đao nấu tôm thịt,Món chính,30phút,,https://vncooking.com/cong-thuc/canh-bi-dao-na...,NaN,NaN,NaN
8,Keyword,3,283,0.176471,Canh tôm nấu bí,Món ngon hàng ngày,,,https://vnexpress.net/canh-tom-nau-bi-4311354....,NaN,NaN,NaN
9,Hybrid_Content,1,394,0.366981,Canh bí ngòi nấu tôm theo công thức của Chef T...,Món ngon hàng ngày,,,https://vnexpress.net/canh-bi-ngoi-nau-tom-the...,NaN,NaN,NaN


Query: Bánh bông lan flan bằng nồi chiên không dầu thơm ngon mềm mịn


...

,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,7644,0.414748,"Cá trắm hấp bia thơm ngon lạ vị, đơn giản dễ l...",Món hấp,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-c...,NaN,NaN,NaN
1,TFIDF,2,1474,0.414106,Canh cá song nấu dứa chua ngọt đậm vị đơn giản...,Món canh,50 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-c...,NaN,NaN,NaN
2,TFIDF,3,1918,0.398209,Bánh mì sữa bắp Nhật Bản thơm ngon mềm mịn,Món bánh,50 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,1972,0.509540,Bánh Castella cupcake bí đỏ thơm ngon mềm mịn,Món bánh,90 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,3253,0.497321,Bánh chocopie siêu dễ thơm ngon tại nhà,Món bánh,120 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,8362,0.484082,Gỏi bưởi thịt gà thơm ngon đơn giản đậm đà hươ...,Món gỏi - salad,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-g...,NaN,NaN,NaN
6,Keyword,1,9151,0.846154,Cách làm pudding xoài cho bé ăn dặm thơm ...,Ăn vặt,240 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-p...,NaN,NaN,NaN
7,Keyword,2,3202,0.350000,Cách làm Pudding phô mai thơm ngon béo mịn...,Món bánh,15 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-p...,NaN,NaN,NaN
8,Keyword,3,6548,0.304348,Cách làm lươn om nước dừa cho bé cho bé ăn ...,Món kho,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-l...,NaN,NaN,NaN
9,Hybrid_Content,1,1972,0.412469,Bánh Castella cupcake bí đỏ thơm ngon mềm mịn,Món bánh,90 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN


Query: Cháo khoai gạo lứt dành cho người đau dạ dày


,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,7486,0.743636,Cháo khoai gạo lứt dành cho người đau dạ dày,Món hấp,40 phút,,https://www.dienmayxanh.com/vao-bep/chao-khoai...,NaN,NaN,NaN
1,TFIDF,2,9602,0.592202,Sữa gạo lứt ngon bổ rẻ vô cùng đơn giản,Thức uống,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-s...,NaN,NaN,NaN
2,TFIDF,3,8971,0.591270,Sữa chua gạo lứt healthy giữ dáng tại nhà bằng...,Ăn vặt,45 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-s...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,9406,0.453458,"Trà gạo lứt đậu đen giảm cân, đẹp da đơn giản ...",Thức uống,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,4981,0.445997,"Cháo lươn bí đỏ cho bé ngon ngọt, hấp dẫn",Món cháo,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-nau-c...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,2333,0.439050,Bánh kem vị cam bưởi sang chảnh thơm ngon bắt ...,Món bánh,60 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-b...,NaN,NaN,NaN
6,Keyword,1,7486,0.666667,Cháo khoai gạo lứt dành cho người đau dạ dày,Món hấp,40 phút,,https://www.dienmayxanh.com/vao-bep/chao-khoai...,NaN,NaN,NaN
7,Keyword,2,5057,0.208333,"Cháo thịt bò khoai tây thơm ngon, bổ dưỡng cho...",Món cháo,35 phút,,https://www.dienmayxanh.com/vao-bep/cach-nau-c...,NaN,NaN,NaN
8,Keyword,3,5102,0.200000,"Cháo gạo lứt giảm cân, bồi bổ sức khỏe cho ngư...",Món cháo,60 phút,,https://www.dienmayxanh.com/vao-bep/cach-nau-c...,NaN,NaN,NaN
9,Hybrid_Content,1,9406,0.456553,"Trà gạo lứt đậu đen giảm cân, đẹp da đơn giản ...",Thức uống,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-t...,NaN,NaN,NaN


Query: Dưa gang muối chua vàng giòn thơm ngon không bị đắng


,method,rank,doc_id,score,title,type_of_food,cook_time,calories,link,max_similarity,min_similarity,num_sentences
0,TFIDF,1,10139,0.406477,Dưa gang muối chua vàng giòn thơm ngon không b...,Món khô - mắm,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-d...,NaN,NaN,NaN
1,TFIDF,2,8707,0.274086,"Cách muối dưa cải (dưa chua) vàng giòn, để lâu...",Món chay,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-d...,NaN,NaN,NaN
2,TFIDF,3,10057,0.220475,Cách muối dưa củ cải giòn ngon không hăng bằng...,Món khô - mắm,5 phút,,https://www.dienmayxanh.com/vao-bep/cach-muoi-...,NaN,NaN,NaN
3,Ingredient_TFIDF,1,9575,0.240916,Công thức pha chế Cocktail Whisky Sour mát lạn...,Thức uống,10 phút,,https://www.dienmayxanh.com/vao-bep/cong-thuc-...,NaN,NaN,NaN
4,Ingredient_TFIDF,2,690,0.239505,Những món chế biến từ hoa bưởi,"Món tráng miệng, giải khát",,,https://vnexpress.net/nhung-mon-che-bien-tu-ho...,NaN,NaN,NaN
5,Ingredient_TFIDF,3,8942,0.237880,"Kẹo muối socola mới lạ, đơn giản, đảm bảo thàn...",Ăn vặt,15 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-k...,NaN,NaN,NaN
6,Keyword,1,10139,0.714286,Dưa gang muối chua vàng giòn thơm ngon không b...,Món khô - mắm,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-d...,NaN,NaN,NaN
7,Keyword,2,8707,0.300000,"Cách muối dưa cải (dưa chua) vàng giòn, để lâu...",Món chay,30 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-d...,NaN,NaN,NaN
8,Keyword,3,10152,0.285714,Cách muối mùng chua Nghệ An vàng giòn thơm ngo...,Món khô - mắm,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-muoi-...,NaN,NaN,NaN
9,Hybrid_Content,1,10139,0.223170,Dưa gang muối chua vàng giòn thơm ngon không b...,Món khô - mắm,20 phút,,https://www.dienmayxanh.com/vao-bep/cach-lam-d...,NaN,NaN,NaN


## 7. Notes for Future Evaluation

For future evaluation, reuse the method functions above but write each result as a run file with `query_id`, `method_name`, `doc_id`, `rank`, and `score`. The important rule is to keep inference query-based:

- TF-IDF methods must use `vectorizer.transform([normalized_query])`.
- SBERT methods must use `model.encode([query])`.
- RARec late fusion must compare the encoded query against each recipe's sentence embeddings and rank by average similarity.
- No method should use `source_doc_id` unless the benchmark is explicitly recipe-to-recipe retrieval.